
# Preprocessing

Inputs: data/listings.csv (Inside Airbnb)

Outputs: 
- data/clean.parquet (Preprocessed data)
- outputs/evidence.json (Data cited in report)

### Section 0: Setup

In [30]:
import json
import os
import pathlib

import numpy as np
import pandas as pd

# Constants:
PROJECT_ROOT = pathlib.Path().resolve().parent
DEFAULT_ROUNDING_DECIMALS = 4
RANDOM_SEED = 20008
CBD_LOCATION = (-37.8183, 144.9671)  # Flinders Street Station in the format (latitude, longitude)
CLEANED_DATA_FILENAME = "clean.parquet"
EVIDENCE_FILENAME = "evidence_preprocess.json"


# Utility function used to round a value to a given number of places
def round_value(value, digits=DEFAULT_ROUNDING_DECIMALS):
    return round(float(value), digits)


# Listings dataframe
listings = pd.read_csv(PROJECT_ROOT / "data" / "listings.csv")

# Evidence dictionary, dumped into evidence.json and cited in the report.
# Keys follow <subject>_<suffix>:
#   _count                   number of rows
#   _share                   proportion, 0..1
#   _pct                     percentage, 0..100
#   _median/_mean/_max/_min  statistic, in the subject's own units
#   _vs_<other>_spearman     rank correlation between two subjects
#   p_<A>_given_<B>          conditional probability, 0..1; order matters
# Subjects are spelled out rather than abbreviated (dsl -> days_since_last_review).
evidence = {"row_count": len(listings), "host_count": listings["host_id"].nunique()}

### Section 1: Parsing text into numbers

In [31]:
# Price parsing
listings["price_num"] = listings["price"].str.replace(r"[$,]", "", regex=True).astype(float)

# Amenity count parsing
listings["amenity_count"] = listings["amenities"].apply(lambda raw_json: len(json.loads(raw_json)))

# Target variable 'y' (whether host is a superhost) parsing
# The result is binary - 1 is yes, 0 is no
listings["y"] = (listings["host_is_superhost"] == "t").astype(int)
evidence["superhost_share_overall"] = round_value(listings["y"].mean())
is_superhost = listings["y"] == 1

# One host has many listings. Calculate evidence regarding
# the number of listings that are under multihosts to help
# determine how the data should be split.
listings_per_host = listings.groupby("host_id").size()
evidence["single_listing_host_count"] = int((listings_per_host == 1).sum())
evidence["multi_listing_host_count"] = int((listings_per_host > 1).sum())
evidence["multi_listing_host_listing_count"] = int(listings_per_host[listings_per_host > 1].sum())
evidence["multi_listing_host_listing_share"] = round_value(evidence["multi_listing_host_listing_count"] / len(listings))

### Section 2: Calculating days since last review and handling zero-review listings

In [32]:
# Handling 'days since last review' column
last_review_date = pd.to_datetime(listings["last_review"], errors="coerce")  # 'coerce' is used to suppress NaN parsing errors
listings["has_review"] = last_review_date.notna().astype(int)
has_review_bool = listings["has_review"] == 1
days_since_review = (last_review_date.max() - last_review_date).dt.days

# Update evidence with relevant information
evidence.update(
    {
        # A reference date that the statistics are based on
        "reference_date": str(last_review_date.max().date()),
        # Describing 'Date since last review' data distribution
        "days_since_last_review_median": round_value(days_since_review[has_review_bool].median()),
        "days_since_last_review_max": int(days_since_review.max()),
        "days_since_last_review_mean_superhost": round_value(days_since_review[has_review_bool & is_superhost].mean(), 2),
        "days_since_last_review_mean_nonsuperhost": round_value(days_since_review[has_review_bool & ~is_superhost].mean(), 2),
        # Size of the zero-review group
        "zero_review_count": int((~has_review_bool).sum()),
        "zero_review_pct": round_value((~has_review_bool).mean() * 100, 2),
        # Calculate superhost status within the zero-review group
        # to determine whether we can drop the relevant rows or not
        "superhost_share_no_review": round_value(listings.loc[~has_review_bool, "y"].mean()),
        "superhost_share_has_review": round_value(listings.loc[has_review_bool, "y"].mean()),
        "superhost_count_no_review": int((~has_review_bool & is_superhost).sum()),
        "superhost_count_no_review_with_reviewed_host": int(listings.loc[~has_review_bool & is_superhost, "host_id"].isin(listings.loc[has_review_bool, "host_id"]).sum()),
        # Days since last review (with reference date) v.s. the feature 'number_of_reviews_ltm'
        # which is the number of reviews in the last 12 months
        "days_since_last_review_vs_reviews_ltm_spearman": round_value(days_since_review[has_review_bool].corr(listings.loc[has_review_bool, "number_of_reviews_ltm"], method="spearman")),
    }
)

# Inputation value
evidence["days_since_last_review_fill_value"] = evidence["days_since_last_review_max"] + 1
listings["days_since_last_review"] = days_since_review.fillna(evidence["days_since_last_review_fill_value"])

### Section 3: Amenity count

In [33]:
# Wrong counting method from assignment 1
def count_amenities_naive(amenities: str):
    if pd.isna(amenities):
        return 0
    text = amenities.strip()
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1]
    if text == "":
        return 0
    return len(text.split(","))


naive_amenity_count = listings["amenities"].apply(count_amenities_naive)
amenity_count_differs = naive_amenity_count != listings["amenity_count"]


evidence.update(
    {
        "amenity_naive_diff_count": int(amenity_count_differs.sum()),
        "amenity_naive_diff_pct": round_value(amenity_count_differs.mean() * 100, 2),
        "amenity_naive_max_diff": int((naive_amenity_count - listings["amenity_count"]).abs().max()),
        "amenity_zero_count": int((listings["amenity_count"] == 0).sum()),
        "amenity_zero_pct": round_value((listings["amenity_count"] == 0).mean() * 100, 2),
        "amenity_count_mean_superhost": round_value(listings.loc[is_superhost, "amenity_count"].mean(), 2),
        "amenity_count_mean_nonsuperhost": round_value(listings.loc[~is_superhost, "amenity_count"].mean(), 2),
    }
)

### Section 4: Missing prices

In [34]:
price_is_missing = listings["price_num"].isna()
is_unavailable_all_year = listings["availability_365"] == 0
price_median = listings["price_num"].median()

# Draw a random 75% of listings and compute the median
# to demonstrate that test-set prices in the computation
# is has negligible effects
rng = np.random.default_rng(RANDOM_SEED)
subsample_medians = [listings["price_num"].iloc[rng.choice(len(listings), int(len(listings) * 0.75), replace=False)].median() for _ in range(10)]

evidence.update(
    {
        "price_missing_count": int(price_is_missing.sum()),
        "price_missing_pct": round_value(price_is_missing.mean() * 100, 2),
        "p_price_missing_given_unavailable": round_value(price_is_missing[is_unavailable_all_year].mean()),
        "p_price_missing_given_available": round_value(price_is_missing[~is_unavailable_all_year].mean()),
        "p_unavailable_given_price_missing": round_value(is_unavailable_all_year[price_is_missing].mean()),
        "superhost_share_unavailable": round_value(listings.loc[is_unavailable_all_year, "y"].mean()),
        "superhost_share_available": round_value(listings.loc[~is_unavailable_all_year, "y"].mean()),
        "price_median_all": round_value(price_median, 2),
        "price_median_subsample_min": round_value(min(subsample_medians), 2),
        "price_median_subsample_max": round_value(max(subsample_medians), 2),
    }
)

listings["price_missing"] = price_is_missing.astype(int)
listings["price_filled"] = listings["price_num"].fillna(price_median)

### Section 5: Distance to the CBD

In [35]:
north_south_km = (listings["latitude"] - CBD_LOCATION[0]) * 111
east_west_km = (listings["longitude"] - CBD_LOCATION[1]) * 111 * np.cos(np.radians(CBD_LOCATION[0]))
listings["dist_cbd"] = np.sqrt(north_south_km**2 + east_west_km**2)

### Section 6: Evidence for rejected candidate steps

In [36]:
bedrooms_is_missing = listings["bedrooms"].isna()
property_type_counts = listings["property_type"].value_counts()
evidence.update(
    {
        "bedrooms_missing_pct": round_value(bedrooms_is_missing.mean() * 100, 2),
        "beds_missing_pct": round_value(listings["beds"].isna().mean() * 100, 2),
        "bedrooms_missing_share_by_room_type": bedrooms_is_missing.groupby(listings["room_type"]).mean().round(4).to_dict(),
        "property_type_category_count": len(property_type_counts),
        "property_type_under_10_count": int((property_type_counts < 10).sum()),
        "property_type_rare_row_count": int(property_type_counts[property_type_counts < 10].sum()),
        "property_type_rare_row_pct": round_value(property_type_counts[property_type_counts < 10].sum() / len(listings) * 100, 2),
        "property_type_top_share_by_room_type": listings.groupby("room_type")["property_type"].agg(lambda types: types.value_counts(normalize=True).iloc[0]).round(4).to_dict(),
        "entire_home_share": round_value((listings["room_type"] == "Entire home/apt").mean()),
        "min_nights_ge30_count": int((listings["minimum_nights"] >= 30).sum()),
        "min_nights_ge30_pct": round_value((listings["minimum_nights"] >= 30).mean() * 100, 2),
    }
)

### Section 7: checking data and exporting

In [37]:
# Row count should not have changed
assert len(listings) == evidence["row_count"]
# At most one distinct 'y' value per host
assert listings.groupby("host_id")["y"].nunique().max() == 1
# No nulls left in day counts and prices
assert listings[["days_since_last_review", "price_filled"]].notna().all().all()

# Save data
listings.to_parquet(PROJECT_ROOT / "data" / CLEANED_DATA_FILENAME, index=False)
os.makedirs(PROJECT_ROOT / "outputs", exist_ok=True)
with open(PROJECT_ROOT / "outputs" / EVIDENCE_FILENAME, "w") as evidence_file:
    json.dump(evidence, evidence_file, indent=2, ensure_ascii=False, default=int)